In [ ]:
using QuantumToolbox, Bessels

In [ ]:
n_a = 12
n_b = 6

ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
φ_b = 0.204

E_J = 37.12*2*π*1.5

##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ε_p = 0.1*π
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 4 *g2

##Env
κ_b=π/10.4;

In [ ]:
# ------------------------------------------------------------
# build static operators once
# ------------------------------------------------------------
a = destroy(n_a)
b = destroy(n_b)

Ia = eye(n_a)
Ib = eye(n_b)

a_tot = tensor(a, Ib)
b_tot = tensor(Ia, b)

na_tot = tensor(a*dag(a), Ib)
nb_tot = tensor(Ia, b*dag(b))

φ_a_op_tot = tensor(
    φ_a * (a + dag(a)),
    Ib,
)

φ_b_op_tot = tensor(
    Ia,
    φ_b * (b + dag(b)),
)

H0 = ω_a * na_tot + ω_b * nb_tot

H_pump = -2*E_J*sin(ε_p)*(sinm(φ_a_op_tot + φ_b_op_tot) - φ_a_op_tot - φ_b_op_tot)
f_drive(p,t) = ε_d*cos(ω_d*t)

H_t = H0 + H_pump + QobjEvo(b_tot + dag(b_tot), f_drive)

jump_op = [sqrt(2 * κ_b) * b_tot];

In [ ]:
T = 2π/ω_b
tsave = LinRange(0.0, T, 2)
ψ_0 = tensor(fock(n_a, 1),fock(n_b, 0))

result_me = mesolve(H_t, ψ_0, tsave, jump_op)

In [ ]:
res = eigsolve_al(H_t, T, jump_op, eigvals = 4, maxiter=1000, eigstol = 1e-9);

In [ ]:
res.values